In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/boston311/bin/python


In [1]:
from pathlib import Path
import pandas as pd

# Locate the repository from either its root or notebooks folder.
current = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "data" / "raw").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find the project's data/raw folder.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project folder:", PROJECT_ROOT)
print("\nCSV files:")
for path in sorted(RAW_DIR.glob("*.csv")):
    print(path.name)

Project folder: /Users/nithyaa/Documents/Hackathon/boston311-crew-planner

CSV files:
lagan_311_open_data_2023_rvsd3.csv
lagan_311_open_data_2024_rvsd3.csv


In [2]:
for year in [2023, 2024]:
    path = RAW_DIR / f"lagan_311_open_data_{year}_rvsd3.csv"
    sample = pd.read_csv(path, nrows=5)

    print(f"\n{year} columns:")
    print(sample.columns.tolist())
    display(sample.head(2))


2023 columns:
['case_enquiry_id', 'open_dt', 'sla_target_dt', 'closed_dt', 'on_time', 'case_status', 'closure_reason', 'case_title', 'subject', 'reason', 'type', 'queue', 'department', 'submitted_photo', 'closed_photo', 'location', 'fire_district', 'pwd_district', 'city_council_district', 'police_district', 'neighborhood', 'neighborhood_services_district', 'ward', 'precinct', 'location_street_name', 'location_zipcode', 'latitude', 'longitude', 'geom_4326', 'source']


,case_enquiry_id,open_dt,sla_target_dt,closed_dt,on_time,case_status,closure_reason,case_title,subject,reason,...,neighborhood,neighborhood_services_district,ward,precinct,location_street_name,location_zipcode,latitude,longitude,geom_4326,source
0,101004616857,2023-01-02 15:50:52.307,2023-01-04 08:30:00,2023-01-03 09:41:31.773,ONTIME,Closed,Case Closed. Closed date : 2023-01-03 09:41:31...,Parking Enforcement,Transportation - Traffic Division,Enforcement & Abandoned Vehicles,...,Fenway / Kenmore / Audubon Circle / Longwood,14,5,0502A,INTERSECTION Jersey St &amp; Peterborough St,NaN,42.343691,-71.097398,0101000020E6100000FD0D6DC53BC651C0FC30C60DFE2B...,Employee Generated
1,101004617727,2023-01-03 11:37:13.827,2023-01-04 11:37:14,2023-01-03 12:14:52.223,ONTIME,Closed,Case Closed. Closed date : 2023-01-03 12:14:52...,Parking Enforcement,Transportation - Traffic Division,Enforcement & Abandoned Vehicles,...,Fenway / Kenmore / Audubon Circle / Longwood,14,Ward 5,0502A,1330 Boylston St,2215.0,42.343940,-71.098960,0101000020E61000004A3C545C55C651C0C8E5D539062C...,Employee Generated



2024 columns:
['case_enquiry_id', 'open_dt', 'sla_target_dt', 'closed_dt', 'on_time', 'case_status', 'closure_reason', 'case_title', 'subject', 'reason', 'type', 'queue', 'department', 'submitted_photo', 'closed_photo', 'location', 'fire_district', 'pwd_district', 'city_council_district', 'police_district', 'neighborhood', 'neighborhood_services_district', 'ward', 'precinct', 'location_street_name', 'location_zipcode', 'latitude', 'longitude', 'geom_4326', 'source']


,case_enquiry_id,open_dt,sla_target_dt,closed_dt,on_time,case_status,closure_reason,case_title,subject,reason,...,neighborhood,neighborhood_services_district,ward,precinct,location_street_name,location_zipcode,latitude,longitude,geom_4326,source
0,101005236041,2024-01-04 10:43:17.097,NaN,2024-01-04 11:23:49.81,ONTIME,Closed,Case Closed. Closed date : Thu Jan 04 11:23:49...,Needle Pickup,Mayor's 24 Hour Hotline,Needle Program,...,Fenway / Kenmore / Audubon Circle / Longwood,14,5,0502A,INTERSECTION Jersey St &amp; Peterborough St,NaN,42.343691,-71.097398,0101000020E6100000FD0D6DC53BC651C0FC30C60DFE2B...,Citizens Connect App
1,101005236307,2024-01-04 12:36:07.337,2024-01-05 12:36:09,2024-01-04 14:55:49.667,ONTIME,Closed,Case Closed. Closed date : Thu Jan 04 14:55:49...,Requests for Street Cleaning,Public Works Department,Street Cleaning,...,Fenway / Kenmore / Audubon Circle / Longwood,14,Ward 5,0502A,115 Jersey St,2215.0,42.342650,-71.097110,0101000020E6100000C8E0DD0C37C651C057A987F4DB2B...,Employee Generated


In [3]:
columns_needed = [
    "case_enquiry_id",
    "open_dt",
    "sla_target_dt",
    "closed_dt",
    "on_time",
    "case_status",
    "case_title",
    "subject",
    "reason",
    "type",
    "department",
    "queue",
    "source",
    "neighborhood",
]

frames = []

for year in [2023, 2024]:
    path = RAW_DIR / f"lagan_311_open_data_{year}_rvsd3.csv"

    frame = pd.read_csv(
        path,
        usecols=columns_needed,
        dtype="string",
    )

    frame["source_year"] = year
    frames.append(frame)

    print(f"{year}: {len(frame):,} rows")

raw_requests = pd.concat(frames, ignore_index=True)

# Remove surrounding spaces without changing department codes.
raw_requests["department"] = (
    raw_requests["department"].str.strip()
)

display(
    pd.crosstab(
        raw_requests["department"],
        raw_requests["source_year"],
        dropna=False,
    )
)

2023: 307,791 rows
2024: 306,756 rows


source_year,2023,2024
department,,
ANML,196,185
BHA,134,129
BPD,147,155
BTDT,75314,81447
BWSC,3645,3406
DISB,21,34
DND,41,43
ECON,1,3
GEN,7466,10747


In [4]:
DEPARTMENTS = {
    "PWDx": "Public Works",
    "BTDT": "Transportation",
    "PARK": "Parks & Recreation",
    "ISD": "Inspectional Services",
}

requests = raw_requests[
    raw_requests["department"].isin(DEPARTMENTS)
].copy()

# Standardize request IDs before checking duplicates.
requests["case_enquiry_id"] = (
    requests["case_enquiry_id"]
    .str.strip()
    .replace("", pd.NA)
)

print(f"Rows in scope: {len(requests):,}")
print(f"Exact duplicate rows: {requests.duplicated().sum():,}")
print(f"Missing request IDs: {requests['case_enquiry_id'].isna().sum():,}")

repeated_ids = (
    requests["case_enquiry_id"].notna()
    & requests["case_enquiry_id"].duplicated(keep=False)
)

print(f"Rows with repeated request IDs: {repeated_ids.sum():,}")

display(
    requests[
        ["open_dt", "sla_target_dt", "closed_dt", "type", "queue"]
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

if repeated_ids.any():
    display(
        requests.loc[repeated_ids]
        .sort_values("case_enquiry_id")
        .head(10)
    )

Rows in scope: 558,663
Exact duplicate rows: 0
Missing request IDs: 0
Rows with repeated request IDs: 0


,missing_count
open_dt,0
sla_target_dt,82143
closed_dt,51814
type,0
queue,0


In [5]:
date_columns = ["open_dt", "sla_target_dt", "closed_dt"]
date_checks = []

for column in date_columns:
    original = requests[column].str.strip().replace("", pd.NA)

    parsed = pd.to_datetime(
        original,
        format="mixed",
        errors="coerce",
    )

    date_checks.append({
        "column": column,
        "missing_before": original.isna().sum(),
        "parsing_failures": (original.notna() & parsed.isna()).sum(),
    })

    requests[column] = parsed

display(pd.DataFrame(date_checks))

# Check that opening dates match their source files.
year_mismatch = (
    requests["open_dt"].notna()
    & requests["open_dt"].dt.year.ne(requests["source_year"])
)

print("Opening year mismatches:", year_mismatch.sum())

print(
    "Closed before opened:",
    (requests["closed_dt"] < requests["open_dt"]).sum(),
)

print(
    "Target before opened:",
    (requests["sla_target_dt"] < requests["open_dt"]).sum(),
)

display(
    requests.groupby("source_year")["open_dt"]
    .agg(["min", "max"])
)

,column,missing_before,parsing_failures
0,open_dt,0,0
1,sla_target_dt,82143,0
2,closed_dt,51814,0


Opening year mismatches: 0
Closed before opened: 0
Target before opened: 0


,min,max
source_year,,
2023,2023-01-01 00:11:22.940,2023-12-31 23:50:00.000
2024,2024-01-01 00:20:52.963,2024-12-31 23:47:03.657


All date checks passed: no parsing failures, year mismatches, or dates before opening. We can now create the monthly demand table.


In [6]:
# Count requests by opening month.
requests["month"] = (
    requests["open_dt"].dt.to_period("M").dt.to_timestamp()
)

monthly_demand = (
    requests
    .groupby(["department", "month"])
    .size()
    .reset_index(name="request_count")
    .sort_values(["department", "month"])
    .reset_index(drop=True)
)

# Validate coverage and totals.
expected_index = pd.MultiIndex.from_product(
    [
        list(DEPARTMENTS),
        pd.date_range("2023-01-01", "2024-12-01", freq="MS"),
    ],
    names=["department", "month"],
)

actual_index = pd.MultiIndex.from_frame(
    monthly_demand[["department", "month"]]
)

missing_months = expected_index.difference(actual_index)

assert len(missing_months) == 0, f"Missing months: {missing_months.tolist()}"
assert len(monthly_demand) == 96
assert monthly_demand["request_count"].sum() == len(requests)

print("Validated: 96 department-month rows.")
display(monthly_demand.head(12))

Validated: 96 department-month rows.


,department,month,request_count
0,BTDT,2023-01-01,5692
1,BTDT,2023-02-01,5316
2,BTDT,2023-03-01,6889
3,BTDT,2023-04-01,6143
4,BTDT,2023-05-01,6737
5,BTDT,2023-06-01,6682
6,BTDT,2023-07-01,5517
7,BTDT,2023-08-01,6473
8,BTDT,2023-09-01,7190
9,BTDT,2023-10-01,7018


In [8]:
#small summary for  forecasting

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "monthly_demand.csv"

monthly_demand.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

print("Saved:", output_path)

Saved: /Users/nithyaa/Documents/Hackathon/boston311-crew-planner/data/processed/monthly_demand.csv


In [9]:
# extract the real request type–department–queue combinations for the intake assistant
routing_catalog_full = (
    requests
    .groupby(["type", "department", "queue"])
    .size()
    .reset_index(name="historical_request_count")
    .sort_values(
        ["department", "historical_request_count"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

# Show the 10 most frequent routes per department.
display(
    routing_catalog_full
    .groupby("department", sort=False)
    .head(10)
)

print("Distinct request types:", requests["type"].nunique())
print("Distinct routing combinations:", len(routing_catalog_full))

,type,department,queue,historical_request_count
0,Parking Enforcement,BTDT,BTDT_Parking Enforcement,119034
1,Abandoned Vehicles,BTDT,BTDT_AVRS Interface Queue,9149
2,Sign Repair,BTDT,BTDT_Sign Shop_Sign Installation,7556
3,Traffic Signal Inspection,BTDT,BTDT_Traffic Signal_Repair,5229
4,Abandoned Bicycle,BTDT,BTDT_Abandoned Bicycle,1968
5,Traffic Signal Inspection,BTDT,BTDT_Engineering_Request for Traffic Signal St...,1417
6,New Sign Crosswalk or Pavement Marking,BTDT,BTDT_Engineering_New Sign and Pavement Marking...,1310
7,Requests for Traffic Signal Studies or Reviews,BTDT,BTDT_Engineering_Request for Traffic Signal St...,959
8,Transportation General Request,BTDT,BTDT_Administration,946
9,Sign Repair,BTDT,BTDT_District 02,627


Distinct request types: 163
Distinct routing combinations: 2038


In [10]:
# Check which request types have multiple routing combinations
route_counts = (
    routing_catalog_full
    .groupby("type")
    .size()
    .reset_index(name="route_count")
)

display(
    route_counts[route_counts["route_count"] > 1]
    .sort_values("route_count", ascending=False)
)

,type,route_count
111,Requests for Street Cleaning,84
125,Sidewalk Repair (Make Safe),78
32,General Comments For a Program or Policy,67
101,Public Works General Request,67
25,Equipment Repair,59
...,...,...
15,Cemetery Maintenance Request,2
128,Snow/Ice Control,2
16,Chronic Dampness/Mold,2
56,Misc. Snow Complaint,2


Some request types have many observed department–queue combinations. That means type alone is insufficient to select a queue reliably. These counts may reflect district-specific queues, transfers, or inconsistent labels—we need to inspect them before deciding.

In [12]:
# Requests for Street Cleaning
street_cleaning_routes = routing_catalog_full[
    routing_catalog_full["type"] == "Requests for Street Cleaning"
].sort_values("historical_request_count", ascending=False)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
):
    display(street_cleaning_routes)

,type,department,queue,historical_request_count
1098,Requests for Street Cleaning,PWDx,PWDx_District 1C: Downtown,5900
1101,Requests for Street Cleaning,PWDx,PWDx_District 03: North Dorchester,4910
1104,Requests for Street Cleaning,PWDx,PWDx_Code Enforcement,3514
1105,Requests for Street Cleaning,PWDx,PWDx_District 07: South Dorchester,3331
1107,Requests for Street Cleaning,PWDx,PWDx_District 10B: Roxbury,2767
1109,Requests for Street Cleaning,PWDx,PWDx_District 05: South Boston,2453
1110,Requests for Street Cleaning,PWDx,PWDx_District 04: Allston/Brighton,2416
1111,Requests for Street Cleaning,PWDx,PWDx_District 1B: North End,2392
1112,Requests for Street Cleaning,PWDx,PWDx_District 09: East Boston,2345
1121,Requests for Street Cleaning,PWDx,PWDx_District 02: Jamaica Plain,1738


In [13]:
#heck how its routes differ between years:
street_cleaning_by_year = (
    requests[
        requests["type"] == "Requests for Street Cleaning"
    ]
    .groupby(["department", "queue", "source_year"])
    .size()
    .unstack("source_year", fill_value=0)
)

street_cleaning_by_year["total"] = (
    street_cleaning_by_year.sum(axis=1)
)

display(
    street_cleaning_by_year
    .sort_values("total", ascending=False)
    .head(20)
)

source_year                                                  2023  2024  total
department queue                                                              
PWDx       PWDx_District 1C: Downtown                        2830  3070   5900
           PWDx_District 03: North Dorchester                2330  2580   4910
           PWDx_Code Enforcement                             1604  1910   3514
           PWDx_District 07: South Dorchester                1627  1704   3331
           PWDx_District 10B: Roxbury                        1381  1386   2767
           PWDx_District 05: South Boston                    1211  1242   2453
           PWDx_District 04: Allston/Brighton                1111  1305   2416
           PWDx_District 1B: North End                       1170  1222   2392
           PWDx_District 09: East Boston                     1200  1145   2345
           PWDx_District 02: Jamaica Plain                    902   836   1738
           PWDx_Missed Trash\Recycling\Yard Waste\Bulk Item   980   648   1628
           PWDx_District 10A: Roxbury                         780   832   1612
           PWDx_District 08: Hyde Park                        793   817   1610
           PWDx_District 06: West Roxbury and Roslindale      753   693   1446
           PWDx_Request for Street Cleaning                   428   323    751
           PWDx_District 1A: Charlestown                      248   275    523
PARK       PARK_Maintenance_Region 1                           68    66    134
PWDx       PWDx_Contractor Complaints                          59    40     99
PARK       PARK_Tree Emergencies                               46    33     79
BTDT       BTDT_BostonBikes                                    43    24     67

In [16]:
# Compare neighborhoods 
street_cleaning = requests[
    requests["type"] == "Requests for Street Cleaning"
].copy()

neighborhood_routes = (
    street_cleaning
    .groupby(
        ["neighborhood", "department", "queue"],
        dropna=False,
    )
    .size()
    .reset_index(name="request_count")
)

# Show the three most frequent routes in each neighborhood.
top_neighborhood_routes = (
    neighborhood_routes
    .sort_values(
        ["neighborhood", "request_count"],
        ascending=[True, False],
    )
    .groupby("neighborhood", dropna=False)
    .head(3)
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
):
    display(top_neighborhood_routes)

,neighborhood,department,queue,request_count
3,Allston,PWDx,PWDx_District 04: Allston/Brighton,151
2,Allston,PWDx,PWDx_Code Enforcement,12
4,Allston,PWDx,PWDx_Missed Trash\Recycling\Yard Waste\Bulk Item,7
23,Allston / Brighton,PWDx,PWDx_District 04: Allston/Brighton,2118
21,Allston / Brighton,PWDx,PWDx_Code Enforcement,262
27,Allston / Brighton,PWDx,PWDx_Missed Trash\Recycling\Yard Waste\Bulk Item,78
54,Back Bay,PWDx,PWDx_District 1C: Downtown,1001
50,Back Bay,PWDx,PWDx_Code Enforcement,304
51,Back Bay,PWDx,PWDx_District 10A: Roxbury,120
81,Beacon Hill,PWDx,PWDx_District 1B: North End,1243


In [18]:
# Keep catalog rows whose type has exactly one routing combination.
routes_per_type = (
    routing_catalog_full
    .groupby("type")["type"]
    .transform("size")
)

simple_types = (
    routing_catalog_full.loc[routes_per_type == 1]
    .copy()
    .sort_values(
        ["department", "historical_request_count"],
        ascending=[True, False],
    )
)

print("Request types with one observed route:", len(simple_types))

with pd.option_context("display.max_colwidth", None):
    display(
        simple_types
        .groupby("department")
        .head(5)
    )

Request types with one observed route: 38


,type,department,queue,historical_request_count
34,Parking Meter Repairs,BTDT,BTDT_Broken Meter Complaints,75
73,Phone Bank Service Inquiry,BTDT,BTDT_OPC_Phone Bank Service,23
105,Valet Parking Problems,BTDT,BTDT_Parking Lot_Valet Parking Problems,14
117,Private Parking Lot Complaints,BTDT,BTDT_Parking Lot_PrivateParkingLotComplaints,11
184,Walk-In Service Inquiry,BTDT,BTDT_OPC_Walk-In Service,5
465,Food Alert - Confirmed,ISD,ISD_Health (INTERNAL),479
470,Food Alert - Unconfirmed,ISD,ISD_Health (INTERNAL),272
473,Cross Metering - Sub-Metering,ISD,ISD_Housing (INTERNAL),223
485,Unsanitary Conditions - Employees,ISD,ISD_Health (INTERNAL),97
508,No Utilities - Food Establishment - Water,ISD,ISD_Health (INTERNAL),31


In [19]:
type_summary = (
    routing_catalog_full
    .groupby(["department", "type"])
    .agg(
        historical_request_count=("historical_request_count", "sum"),
        queue_count=("queue", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["department", "historical_request_count"],
        ascending=[True, False],
    )
)

with pd.option_context(
    "display.max_rows", 40,
    "display.max_colwidth", None,
):
    display(
        type_summary
        .groupby("department")
        .head(8)
        .reset_index(drop=True)
    )

,department,type,historical_request_count,queue_count
0,BTDT,Parking Enforcement,119597,20
1,BTDT,Sign Repair,10177,22
2,BTDT,Abandoned Vehicles,9312,6
3,BTDT,Traffic Signal Inspection,6977,23
4,BTDT,Abandoned Bicycle,2184,9
5,BTDT,New Sign Crosswalk or Pavement Marking,1749,19
6,BTDT,Requests for Traffic Signal Studies or Reviews,1230,13
7,BTDT,Transportation General Request,1133,17
8,ISD,Rodent Activity,8341,6
9,ISD,Unsatisfactory Living Conditions,5167,6


In [20]:
selected_types = [
    "Parking Enforcement",
    "Sign Repair",
    "Abandoned Vehicles",
    "Traffic Signal Inspection",
    "Rodent Activity",
    "Unsatisfactory Living Conditions",
    "Pest Infestation - Residential",
    "Heat - Excessive Insufficient",
    "Tree Maintenance Requests",
    "Ground Maintenance",
    "Equipment Repair",
    "New Tree Requests",
    "Mattress_Pickup",
    "Schedule a Bulk Item Pickup",
    "Missed Trash/Recycling/Yard Waste/Bulk Item",
    "Request for Pothole Repair",
]

normalized_types = (
    routing_catalog_full["type"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

selected_routes = routing_catalog_full[
    normalized_types.isin(selected_types)
].copy()

matched_types = set(
    normalized_types[normalized_types.isin(selected_types)]
)

assert not (set(selected_types) - matched_types), (
    f"Unmatched types: {set(selected_types) - matched_types}"
)

# Inspect the three most frequent routes per type.
route_preview = (
    selected_routes
    .sort_values(
        ["type", "historical_request_count"],
        ascending=[True, False],
    )
    .groupby("type")
    .head(3)
)

with pd.option_context(
    "display.max_rows", 60,
    "display.max_colwidth", None,
):
    display(route_preview.reset_index(drop=True))
    

,type,department,queue,historical_request_count
0,Abandoned Vehicles,BTDT,BTDT_AVRS Interface Queue,9149
1,Abandoned Vehicles,BTDT,BTDT_Parking Enforcement,148
2,Abandoned Vehicles,PWDx,PWDx_Code Enforcement,12
3,Equipment Repair,PARK,PARK_Maintenance_Trades,953
4,Equipment Repair,PARK,PARK_Carpentry,833
5,Equipment Repair,PARK,PARK_Plumbing,810
6,Ground Maintenance,PARK,PARK_Maintenance_Region 1,1370
7,Ground Maintenance,PARK,PARK_Maintenance_Region 2,1229
8,Ground Maintenance,PARK,PARK_Maintenance_Region 6,1107
9,Heat - Excessive Insufficient,ISD,ISD_Housing (INTERNAL),1665


In [21]:
# Work on a copy and add a consistent label for grouping.
catalog = selected_routes.copy()

catalog["type_label"] = (
    catalog["type"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Calculate each route's share of all selected records for its type.
type_totals = (
    catalog.groupby("type_label")["historical_request_count"]
    .transform("sum")
)

catalog["historical_route_share"] = (
    catalog["historical_request_count"] / type_totals
)

catalog = catalog.sort_values(
    ["type_label", "historical_request_count", "department", "queue"],
    ascending=[True, False, True, True],
).reset_index(drop=True)

# Keep every route, not just the top-three preview.
catalog.to_csv(
    PROCESSED_DIR / "routing_catalog_selected.csv",
    index=False,
)

# Review the leading route and its share for each type.
routing_review = (
    catalog.drop_duplicates("type_label")
    [[
        "type_label",
        "department",
        "queue",
        "historical_request_count",
        "historical_route_share",
    ]]
    .rename(columns={
        "department": "leading_department",
        "queue": "leading_queue",
    })
    .copy()
)

routing_review["historical_route_share"] = (
    routing_review["historical_route_share"].mul(100).round(1)
)

with pd.option_context("display.max_colwidth", None):
    display(routing_review.reset_index(drop=True))

,type_label,leading_department,leading_queue,historical_request_count,historical_route_share
0,Abandoned Vehicles,BTDT,BTDT_AVRS Interface Queue,9149,98.0
1,Equipment Repair,PARK,PARK_Maintenance_Trades,953,18.9
2,Ground Maintenance,PARK,PARK_Maintenance_Region 1,1370,20.5
3,Heat - Excessive Insufficient,ISD,ISD_Housing (INTERNAL),1665,99.8
4,Mattress_Pickup,PWDx,PWDx_WM_Mattress_Pickup,36546,100.0
5,Missed Trash/Recycling/Yard Waste/Bulk Item,PWDx,PWDx_Missed Trash\Recycling\Yard Waste\Bulk Item,22932,99.0
6,New Tree Requests,PARK,PARK_New Tree Requests,4376,90.1
7,Parking Enforcement,BTDT,BTDT_Parking Enforcement,119034,99.4
8,Pest Infestation - Residential,ISD,ISD_Environmental Services (INTERNAL),2001,58.9
9,Request for Pothole Repair,PWDx,PWDx_Contractor Complaints,2290,13.4


In [22]:
# routing_review stores percentages on a 0–100 scale.
routing_review["routing_policy"] = (
    routing_review["historical_route_share"]
    .ge(90)
    .map({
        True: "historical_suggestion",
        False: "context_review_required",
    })
)

routing_review.to_csv(
    PROCESSED_DIR / "routing_review.csv",
    index=False,
)

print("Saved routing_review.csv")

Saved routing_review.csv
